#Introducción

##Objetivo del notebook

En este notebook se realiza la carga, exploración inicial y limpieza básica del dataset público argentino de demanda eléctrica anual obtenido de la Subsecretaría de Programación Macroeconómica.

El foco está en:

- Comprender la estructura del dataset

- Identificar tipos de datos y posibles problemas (valores nulos, escalas, unidades)

- Seleccionar las variables relevantes para el análisis y para la posterior construcción de documentos semánticos

Este notebook no construye IA ni embeddings. Su función es garantizar que los datos crudos sean comprensibles y confiables antes de ser transformados en conocimiento.


Datos crudos

   ↓

Datos limpios (01)

   ↓

Documentos semánticos (02)

   ↓

Vector store (03)

   ↓

Consultas y análisis (04)


In [38]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

ruta_csv = "/content/drive/MyDrive/Energia_Rag/demanda_electricidad_anual.csv"

df = pd.read_csv(ruta_csv)

df.sample(5)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,indice_tiempo,demanda_total,demanda_residencial,comercio_e_industria,grandes_usuarios,temperatura_promedio,potencia_maxima,ede_tucuman,edelap_sa,edenor_distribuidor,...,potencia_instalada_ciclos_combinados,potencia_instalada_turbina_a_gas,potencia_instalada_turbovapor,potencia_instalada_nuclear,potencia_instalada_motor_diesel,potencia_instalada_eolica,potencia_instalada_biogas,potencia_instalada_solar,potencia_instalada_hidraulica_renovable,potencia_instalada_total
3,2004-01-01,87172.410000,NaN,NaN,NaN,18.442667,15032.000000,NaN,NaN,NaN,...,6362.670,2317.200,4526.0,1005.0,4.000,0.000,0.000,0.00,381.29,23953.390
0,2001-01-01,78101.854000,NaN,NaN,NaN,18.485333,14061.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,2022-01-01,138775.233404,63156.982337,51661.950967,23956.300100,18.416667,24033.583333,2856.428654,2913.781006,23458.021572,...,13499.544,5827.801,4251.2,1755.0,1696.448,3309.315,142.895,1085.83,524.31,42926.723
22,2023-01-01,140883.538343,65286.370174,51727.537554,23869.630615,19.400000,24363.416667,2992.843006,2945.583410,23834.970922,...,14235.004,5290.571,4251.2,1755.0,1660.398,3705.415,151.433,1365.94,524.31,43773.651
15,2016-01-01,133110.794625,57067.277429,51897.432639,24146.084557,18.100000,25380.000000,2939.882235,2816.603973,24457.502373,...,9227.130,5251.490,4451.2,1755.0,1834.247,187.350,16.600,8.20,488.24,33971.157


In [39]:
df.describe()
df.columns

Index(['indice_tiempo', 'demanda_total', 'demanda_residencial',
       'comercio_e_industria', 'grandes_usuarios', 'temperatura_promedio',
       'potencia_maxima', 'ede_tucuman', 'edelap_sa', 'edenor_distribuidor',
       'edesal_distribuidor', 'edestesa_emp_dist__este', 'edesur_distribuidor',
       'emp_de_energia_de_rio_negro_sa', 'emp_dist_energ_atlantica',
       'emp_de_energia_de_la_rioja_sa', 'emp_dist_energia_de_salta',
       'emp_electric_misiones_sa', 'empresa_dis_estero_sa',
       'empresa_jujenia_de_energia_sa', 'energia_de_catamarca_sa',
       'energia_de_entre_rios_sa', 'energia_de_mendoza_sa',
       'energia_san_juan_sa_exedessa', 'epec_distribuidor',
       'epen_distribuidor', 'epesf_distribuidor',
       'recursos_y_energia_formosa_sa', 'secheep', 'spse_santa_cruz',
       'potencia_instalada_hidraulica', 'potencia_instalada_ciclos_combinados',
       'potencia_instalada_turbina_a_gas', 'potencia_instalada_turbovapor',
       'potencia_instalada_nuclear', 'poten

Por el momento se excluiran distintas columnas para evitar ruido y dificultar el proceso rag, se mantendran:

- indice_tiempo
- demanda_total
- demanda_residencial
- comercio_e_industria
- grandes_usuarios
- temperatura_promedio
- potencia_maxima


In [40]:
columnas_rag= ['indice_tiempo','demanda_total', 'demanda_residencial', 'comercio_e_industria','grandes_usuarios', 'temperatura_promedio','potencia_maxima']
df_rag=df[columnas_rag].copy()
df_rag.isnull().sum()
df_rag.head()
#df_rag['indice_tiempo'].value_counts()

df_rag['indice_tiempo']=pd.to_datetime(df_rag['indice_tiempo'])
df_rag.dtypes
df_rag.head()

,indice_tiempo,demanda_total,demanda_residencial,comercio_e_industria,grandes_usuarios,temperatura_promedio,potencia_maxima
0,2001-01-01,78101.854,NaN,NaN,NaN,18.485333,14061.0
1,2002-01-01,76472.890,NaN,NaN,NaN,18.289667,13481.0
2,2003-01-01,82213.370,NaN,NaN,NaN,17.816000,14359.0
3,2004-01-01,87172.410,NaN,NaN,NaN,18.442667,15032.0
4,2005-01-01,92332.950,30086.495,43102.875,19143.58,18.363667,16143.0


En el presente dataframe se contempla el periodo comprendido del 01-01-2001 al 01-01-2024. En el cual el primer periodo de 4 años no presenta datos en demanda residencial, comercio e industria, ni grandes usuarios.

In [41]:
df_rag.to_csv("/content/drive/MyDrive/Energia_Rag/df_rag.csv", index=False)
#exportar csv para utilizacion en 2do notebook